Data setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('/content/creditcard.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1986 entries, 0 to 1985
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    1986 non-null   int64  
 1   V1      1986 non-null   float64
 2   V2      1986 non-null   float64
 3   V3      1986 non-null   float64
 4   V4      1986 non-null   float64
 5   V5      1986 non-null   float64
 6   V6      1986 non-null   float64
 7   V7      1986 non-null   float64
 8   V8      1986 non-null   float64
 9   V9      1986 non-null   float64
 10  V10     1986 non-null   float64
 11  V11     1986 non-null   float64
 12  V12     1986 non-null   float64
 13  V13     1986 non-null   float64
 14  V14     1985 non-null   float64
 15  V15     1985 non-null   float64
 16  V16     1985 non-null   float64
 17  V17     1985 non-null   float64
 18  V18     1985 non-null   float64
 19  V19     1985 non-null   float64
 20  V20     1985 non-null   float64
 21  V21     1985 non-null   float64
 22  

In [4]:
df.isnull()
df.isnull().sum()

,0
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


In [5]:
df = df.dropna()

Assgining values

In [6]:
x = df.drop('Class', axis = 1)
y = df['Class']

In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0)

Model

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [9]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Support Vector Machine': SVC()
}

In [10]:
from sklearn.metrics import accuracy_score

for name,model in models.items():
  # train
  model.fit(x_train, y_train)
  # prediction
  y_pred = model.predict(x_test)
  # evaluation
  accuracy = accuracy_score(y_test, y_pred)
  print(f'{name} accuracy: {accuracy}')

Logistic Regression accuracy: 1.0
Random Forest accuracy: 1.0
Decision Tree accuracy: 0.9949622166246851
Support Vector Machine accuracy: 1.0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1985 entries, 0 to 1984
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    1985 non-null   int64  
 1   V1      1985 non-null   float64
 2   V2      1985 non-null   float64
 3   V3      1985 non-null   float64
 4   V4      1985 non-null   float64
 5   V5      1985 non-null   float64
 6   V6      1985 non-null   float64
 7   V7      1985 non-null   float64
 8   V8      1985 non-null   float64
 9   V9      1985 non-null   float64
 10  V10     1985 non-null   float64
 11  V11     1985 non-null   float64
 12  V12     1985 non-null   float64
 13  V13     1985 non-null   float64
 14  V14     1985 non-null   float64
 15  V15     1985 non-null   float64
 16  V16     1985 non-null   float64
 17  V17     1985 non-null   float64
 18  V18     1985 non-null   float64
 19  V19     1985 non-null   float64
 20  V20     1985 non-null   float64
 21  V21     1985 non-null   float64
 22  V22  

LSTM branch

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [13]:
x_train_lstm = x_train.values.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test_lstm = x_test.values.reshape((x_test.shape[0], x_test.shape[1], 1))

In [14]:
model = Sequential()
model.add(LSTM(units = 64, return_sequences = True, input_shape = (x_train.shape[1], 1), activation='tanh'))
model.add(Dropout(0.2))
model.add(LSTM(units = 32, activation='tanh'))
model.add(Dropout(0.2))
model.add(Dense(units = 1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [16]:
# class weights
fraud_count  = (y_train == 1).sum()
normal_count = (y_train == 0).sum()
class_weight = {0: 1.0, 1: normal_count / fraud_count}

In [17]:
history = model.fit(x_train_lstm, y_train,
                    epochs=10,
                    batch_size=256,
                    validation_split=0.1,
                    class_weight=class_weight,
                    verbose=1)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 82ms/step - accuracy: 0.9321 - loss: 1.5163 - val_accuracy: 1.0000 - val_loss: 0.6185
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9937 - loss: 1.5101 - val_accuracy: 1.0000 - val_loss: 0.6205
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9846 - loss: 1.4339 - val_accuracy: 0.9937 - val_loss: 0.6172
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9909 - loss: 1.4051 - val_accuracy: 0.9937 - val_loss: 0.6015
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9783 - loss: 1.3940 - val_accuracy: 0.9748 - val_loss: 0.5983
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9776 - loss: 1.3778 - val_accuracy: 0.9686 - val_loss: 0.5757
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9685 - loss: 1.2491 - val_accuracy: 0.9497 - val_loss: 0.5771
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9370 - loss: 1.1280 - val_accuracy: 0.9497 - val_loss: 0.5395


Evaluation

In [18]:
from sklearn.metrics import classification_report

y_pred_lstm = (model.predict(x_test_lstm) > 0.5).astype(int)
accuracy    = (y_pred_lstm.flatten() == y_test).mean()
print(f'LSTM accuracy: {accuracy}')
print(classification_report(y_test, y_pred_lstm, target_names=['Normal', 'Fraud']))

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
LSTM accuracy: 0.947103274559194
              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97       397
       Fraud       0.00      0.00      0.00         0

    accuracy                           0.95       397
   macro avg       0.50      0.47      0.49       397
weighted avg       1.00      0.95      0.97       397



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [20]:
get_ipython().system('git clone https://github.com/neural-shubh/credit-card-fraud.git')
get_ipython().system('cd credit-card-fraud')

Cloning into 'credit-card-fraud'...
fatal: could not read Username for 'https://github.com': No such device or address
/bin/bash: line 1: cd: credit-card-fraud: No such file or directory
